# Day 02 — Conditions and Loops
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** if/elif/else · comparison operators · boolean operators · `in` operator · for loops · enumerate · accumulation pattern · while loops · exponential backoff · break · continue · list comprehension · dict comprehension

> **Connection to Module 00:** The routing rules you wrote by hand in `## Task` — *"Route TRACK_ORDER queries to logistics"* — become `if/elif/else` in Python today.


---


## 1 — if / elif / else

A **condition** is a question your code asks at runtime. The answer is always `True` or `False`.

```python
if condition:          # runs when True
    ...
elif other_condition:  # runs when first was False, this is True
    ...
else:                  # runs when nothing above matched — always write this
    ...
```

**Three rules:**
1. Colon `:` is required at the end of every `if` / `elif` / `else` line
2. Python checks **top to bottom** and **stops at the first match**
3. Always write `else` — real data always surprises you


In [ ]:
# Simple — grade from a score
score = 72

if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
elif score >= 70:
    grade = "C"
else:
    grade = "F"

print(f"score={score} → grade={grade}")


### Comparison and Boolean Operators

| Operator | Meaning | Example |
|----------|---------|--------|
| `==` | equal to | `status == "Delivered"` |
| `!=` | not equal | `status != "Cancelled"` |
| `>` | greater than | `rating > 3` |
| `<` | less than | `price < 100.0` |
| `>=` | greater or equal | `rating >= 4` |
| `<=` | less or equal | `stock <= 0` |
| `in` | substring / membership | `"cancel" in query.lower()` |
| `and` | both must be True | `is_in_stock and is_recommended` |
| `or` | at least one True | `"cancel" in q or "refund" in q` |
| `not` | flip True↔False | `not is_cancelled` |


In [ ]:
price  = 205.21
stock  = 238
rating = 4

is_in_stock    = stock > 0
is_affordable  = price < 300.0
is_recommended = rating >= 4

print(is_in_stock, is_affordable, is_recommended)

if is_in_stock and is_recommended:
    recommendation = "Great choice — in stock and highly rated!"
elif is_in_stock and not is_recommended:
    recommendation = "In stock but reviews are mixed."
else:
    recommendation = "Out of stock."

print(recommendation)


### ShopSmart — Query Routing

In Module 00 you wrote: *"Route TRACK_ORDER queries to logistics."*  
Here those same rules become Python `if/elif/else`.

**Key pattern:** always call `.lower()` once at the top so matching is case-insensitive.  
Put the **most urgent condition first** — cancel/refund before general tracking.


In [ ]:
customer_query = "Where is my order #3042?"
q = customer_query.lower()   # normalise once, use everywhere

if "cancel" in q or "refund" in q:
    agent, urgency = "human_agent", "high"
elif "track" in q or "where" in q or "delivery" in q:
    agent, urgency = "order_agent", "normal"
elif "return" in q or "exchange" in q:
    agent, urgency = "returns_agent", "normal"
elif "price" in q or "discount" in q:
    agent, urgency = "promotions_agent", "normal"
elif "product" in q or "spec" in q:
    agent, urgency = "catalog_agent", "low"
else:
    agent, urgency = "general_agent", "normal"

print(f"Query  : {customer_query}")
print(f"Agent  : {agent}")
print(f"Urgency: {urgency}")


In [ ]:
# Map database status values → customer-facing messages
# The else handles any unexpected value — real data is always messy
statuses = ["Delivered", "In Transit", "Cancelled", "Refunded", "Unknown"]

for status in statuses:
    if status == "Delivered":
        msg = "Your order has been delivered."
    elif status == "In Transit":
        msg = "Your order is on its way."
    elif status == "Processing":
        msg = "Your order is being prepared."
    elif status == "Cancelled":
        msg = "Your order has been cancelled."
    elif status == "Refunded":
        msg = "Your refund has been processed."
    else:
        msg = f"Status '{status}' — please contact support."
    print(f"{status:12s} → {msg}")


---


## 2 — for Loops

A `for` loop runs a block of code **once for each item** in a collection.

```python
for item in collection:
    do something with item
```

| | `for` | `while` |
|-|-------|---------|
| Use when | You have a known collection | You don't know how many iterations |
| Ends when | Collection is exhausted | Condition becomes False |


In [ ]:
fruits = ["apple", "banana", "cherry"]

# ITERATION TRACE:
# Iter 1 → fruit = "apple"   → print "apple"
# Iter 2 → fruit = "banana"  → print "banana"
# Iter 3 → fruit = "cherry"  → print "cherry"
# collection exhausted → loop ends automatically

for fruit in fruits:
    print(fruit)


In [ ]:
# enumerate() — gives index AND value together
# Without enumerate:  for fruit in fruits     → value only
# With enumerate:     for i, fruit in ...      → index + value

# ITERATION TRACE:
# Iter 1 → i=0, fruit="apple"   → print "[0] apple"
# Iter 2 → i=1, fruit="banana"  → print "[1] banana"
# Iter 3 → i=2, fruit="cherry"  → print "[2] cherry"
# collection exhausted → loop ends

for i, fruit in enumerate(fruits):
    print(f"[{i}] {fruit}")


### Accumulation Pattern

Start with an empty list → loop → append one result per iteration.  
This pattern appears in almost every LLM pipeline.

```python
results = []                # start empty
for item in collection:
    results.append(...)     # add one result per iteration
```


In [ ]:
test_queries = [
    "Where is my order #3042?",
    "I want to cancel my purchase",
    "Do you have a discount code?",
    "What are the specs of the Classic Monitor?",
]

results = []   # accumulator — starts empty

# ITERATION TRACE:
# Iter 1 → i=0, query="Where is my order #3042?"
#           q = "where is my order #3042?"
#           "cancel" in q? No | "where" in q? Yes → agent = "order_agent"
#           results = [{index:0, query:"Where...", agent:"order_agent"}]
#
# Iter 2 → i=1, query="I want to cancel my purchase"
#           q = "i want to cancel my purchase"
#           "cancel" in q? Yes → agent = "human_agent"  (stops here — first match)
#           results = [{...order_agent}, {index:1, agent:"human_agent"}]
#
# Iter 3 → i=2, query="Do you have a discount code?"
#           q = "do you have a discount code?"
#           "cancel"? No | "where"? No | "discount"? Yes → agent = "promotions_agent"
#           results = [{...}, {...}, {index:2, agent:"promotions_agent"}]
#
# Iter 4 → i=3, query="What are the specs of the Classic Monitor?"
#           q = "what are the specs of the classic monitor?"
#           "cancel"? No | "where"? No | "discount"? No | "spec"? Yes → agent = "catalog_agent"
#           results = [{...}, {...}, {...}, {index:3, agent:"catalog_agent"}]
# collection exhausted → loop ends

for i, query in enumerate(test_queries):
    q = query.lower()
    if   "cancel" in q or "refund"   in q: agent = "human_agent"
    elif "where"  in q or "track"    in q: agent = "order_agent"
    elif "discount" in q or "price"  in q: agent = "promotions_agent"
    elif "spec"   in q or "product"  in q: agent = "catalog_agent"
    else:                                   agent = "general_agent"

    results.append({"index": i, "query": query, "agent": agent})

for r in results:
    print(f"[{r['index']}] {r['query'][:38]:38s} → {r['agent']}")


In [ ]:
reviews = [
    {"review_id": 5001, "rating": 3, "title": "Its fine"},
    {"review_id": 5002, "rating": 5, "title": "Excellent!"},
    {"review_id": 5003, "rating": 4, "title": "Really good"},
    {"review_id": 5004, "rating": 2, "title": "Disappointed"},
]

# ITERATION TRACE:
# Iter 1 → review = {5001, rating=3, "Its fine"}
#           stars = "★"*3 + "☆"*2 = "★★★☆☆"
#           print "ID:5001  ★★★☆☆  Its fine"
#
# Iter 2 → review = {5002, rating=5, "Excellent!"}
#           stars = "★"*5 + "☆"*0 = "★★★★★"
#           print "ID:5002  ★★★★★  Excellent!"
#
# Iter 3 → review = {5003, rating=4, "Really good"}
#           stars = "★"*4 + "☆"*1 = "★★★★☆"
#           print "ID:5003  ★★★★☆  Really good"
#
# Iter 4 → review = {5004, rating=2, "Disappointed"}
#           stars = "★"*2 + "☆"*3 = "★★☆☆☆"
#           print "ID:5004  ★★☆☆☆  Disappointed"
# collection exhausted → loop ends

for review in reviews:
    stars = "★" * review["rating"] + "☆" * (5 - review["rating"])
    print(f"ID:{review['review_id']}  {stars}  {review['title']}")


---


## 3 — range()

`range()` generates a sequence of integers without storing them all in memory.

| Form | Produces |
|------|----------|
| `range(stop)` | 0, 1, 2, ..., stop-1 |
| `range(start, stop)` | start, start+1, ..., stop-1 |
| `range(start, stop, step)` | every step-th value from start |


In [ ]:
print(list(range(5)))           # [0, 1, 2, 3, 4]
print(list(range(1, 6)))        # [1, 2, 3, 4, 5]
print(list(range(0, 10, 2)))    # [0, 2, 4, 6, 8]
print(list(range(10, 0, -1)))   # [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]

# Batch processing — split 9 queries into batches of 3
total_queries = 9
batch_size    = 3

# range(9 // 3) = range(3) → values: 0, 1, 2
#
# ITERATION TRACE:
# Iter 1 → batch_num=0
#           start = 0 * 3 = 0
#           end   = 0 + 3 = 3
#           print "Batch 1: queries 0 to 2"
#
# Iter 2 → batch_num=1
#           start = 1 * 3 = 3
#           end   = 3 + 3 = 6
#           print "Batch 2: queries 3 to 5"
#
# Iter 3 → batch_num=2
#           start = 2 * 3 = 6
#           end   = 6 + 3 = 9
#           print "Batch 3: queries 6 to 8"
# range(3) exhausted → loop ends

for batch_num in range(total_queries // batch_size):
    start = batch_num * batch_size
    end   = start + batch_size
    print(f"Batch {batch_num + 1}: queries {start} to {end - 1}")


---


## 4 — while Loops

A `while` loop runs **as long as its condition is True**.  
Use it when you don't know in advance how many iterations you need.

```python
while condition:
    do something
    update the condition   # REQUIRED — forget this = infinite loop
```

> **Critical rule:** always update whatever the condition checks inside the loop body.


In [ ]:
count = 0

# ITERATION TRACE:
# Check: count=0  →  0 < 5 = True  → print 0,  count becomes 1
# Check: count=1  →  1 < 5 = True  → print 1,  count becomes 2
# Check: count=2  →  2 < 5 = True  → print 2,  count becomes 3
# Check: count=3  →  3 < 5 = True  → print 3,  count becomes 4
# Check: count=4  →  4 < 5 = True  → print 4,  count becomes 5
# Check: count=5  →  5 < 5 = False → loop exits
# Final count = 5  (one more than the last printed value)

while count < 5:
    print(f"count = {count}")
    count += 1    # update the condition — without this: infinite loop

print(f"Loop ended. Final count = {count}")


### LLM API Retry — Exponential Backoff

LLM APIs return `HTTP 429 (Too Many Requests)` when you call them too fast.  
The production pattern: **wait and retry**, doubling the wait each time.

```
Attempt 1 fails → wait 0.1s
Attempt 2 fails → wait 0.2s
Attempt 3 fails → wait 0.4s
Attempt 4 fails → wait 0.8s
```

`while` is the right tool here — you don't know which attempt will succeed.


In [ ]:
import time

def call_llm_api(query):
    """Simulates: fails first 2 calls, succeeds on 3rd."""
    call_llm_api.count = getattr(call_llm_api, 'count', 0) + 1
    if call_llm_api.count < 3:
        raise Exception("HTTP 429: Rate limit exceeded")
    return "[LLM] Order #3042 is In Transit, arriving Friday."

max_retries = 4
attempt     = 0
response    = None

# ITERATION TRACE:
# Check: attempt=0  →  0 < 4 = True
#        call_llm_api() → raises HTTP 429  (count=1 < 3)
#        wait = 0.1 * (2**0) = 0.1 * 1 = 0.1s
#        attempt = 1
#
# Check: attempt=1  →  1 < 4 = True
#        call_llm_api() → raises HTTP 429  (count=2 < 3)
#        wait = 0.1 * (2**1) = 0.1 * 2 = 0.2s
#        attempt = 2
#
# Check: attempt=2  →  2 < 4 = True
#        call_llm_api() → returns response  (count=3, 3 >= 3)
#        break → exit loop immediately
#        response = "[LLM] Order #3042 is In Transit..."
#
# attempt=3 never checked — break already exited the loop

while attempt < max_retries:
    try:
        response = call_llm_api("Where is order #3042?")
        print(f"  Attempt {attempt + 1}: SUCCESS")
        break                               # success — exit loop
    except Exception as e:
        wait = 0.1 * (2 ** attempt)         # 0.1s → 0.2s → 0.4s → 0.8s
        print(f"  Attempt {attempt + 1}: FAILED ({e}). Waiting {wait:.1f}s")
        time.sleep(wait)
        attempt += 1

print(f"Response: {response}")


---


## 5 — break and continue

| Keyword | Effect | Mental model |
|---------|--------|--------------|
| `break` | Exit the loop entirely right now | "I am done — stop the loop" |
| `continue` | Skip this iteration, go to next | "Not this one — move on" |


In [ ]:
# continue — skip even numbers
# TRACE: n=1 → 1%2=1 ≠ 0 → append  | n=2 → 2%2=0 → continue (skip)
#        n=3 → 3%2=1 ≠ 0 → append  | n=4 → 4%2=0 → continue (skip) ...
odds = []
for n in range(1, 10):
    if n % 2 == 0:
        continue        # even → skip this iteration, jump to n+1
    odds.append(n)
print("Odd numbers:", odds)  # [1, 3, 5, 7, 9]

# break — stop at first number above 5
# TRACE: n=1 → 1>5? No → print | n=2 → 2>5? No → print | ...
#        n=5 → 5>5? No → print  | n=6 → 6>5? Yes → BREAK → n=7,8,9 never reached
for n in range(1, 10):
    if n > 5:
        print(f"Stopped at {n}")  # prints 6, then exits
        break
    print(n, end=" ")             # prints 1 2 3 4 5
print()


### ShopSmart — Auto-select Few-shot Examples

In Module 00 Technique 02 you chose few-shot examples by hand.  
Here Python selects them automatically:
- `continue` → skip reviews with rating < 4
- `break` → stop once we have 3 examples collected


In [ ]:
all_reviews = [
    {"review_id": 5001, "rating": 3, "title": "Its fine"},
    {"review_id": 5002, "rating": 5, "title": "Excellent!"},
    {"review_id": 5003, "rating": 4, "title": "Really good"},
    {"review_id": 5004, "rating": 2, "title": "Disappointed"},
    {"review_id": 5005, "rating": 5, "title": "Perfect product"},
    {"review_id": 5006, "rating": 4, "title": "Solid choice"},
]

good_reviews   = []   # accumulator — starts empty
max_to_collect = 3

# ─────────────────────────────────────────────────────────────
# ITERATION TRACE
#
# Iter 1 → review_id=5001  rating=3
#          rating < 4 ?  3 < 4 = True  → continue  (skip, go to Iter 2)
#          good_reviews = []
#
# Iter 2 → review_id=5002  rating=5
#          rating < 4 ?  5 < 4 = False → do NOT skip
#          good_reviews.append(5002)
#          good_reviews = [{5002, 5, 'Excellent!'}]
#          len(good_reviews) >= 3 ?  1 >= 3 = False → keep looping
#
# Iter 3 → review_id=5003  rating=4
#          rating < 4 ?  4 < 4 = False → do NOT skip
#          good_reviews.append(5003)
#          good_reviews = [{5002}, {5003}]
#          len(good_reviews) >= 3 ?  2 >= 3 = False → keep looping
#
# Iter 4 → review_id=5004  rating=2
#          rating < 4 ?  2 < 4 = True  → continue  (skip, go to Iter 5)
#          good_reviews = [{5002}, {5003}]  (unchanged)
#
# Iter 5 → review_id=5005  rating=5
#          rating < 4 ?  5 < 4 = False → do NOT skip
#          good_reviews.append(5005)
#          good_reviews = [{5002}, {5003}, {5005}]
#          len(good_reviews) >= 3 ?  3 >= 3 = True  → BREAK
#          review_id=5006 is never reached
# ─────────────────────────────────────────────────────────────

for review in all_reviews:

    if review["rating"] < 4:
        continue                     # rating too low — skip to next iteration

    good_reviews.append(review)      # rating >= 4 — keep it

    if len(good_reviews) >= max_to_collect:
        break                        # collected enough — exit loop

# Print results
for r in good_reviews:
    print(f"  {r['review_id']}  {'★' * r['rating']}  {r['title']}")

print()
print("Selected titles:", [r["title"] for r in good_reviews])
print(f"review_id=5006 '{all_reviews[5]['title']}' was never reached — loop broke at iter 5")


**What just happened — iteration by iteration:**

| Iter | review_id | rating | `rating < 4` | Action | `good_reviews` size |
|------|-----------|--------|-------------|--------|---------------------|
| 1 | 5001 | 3 | `3 < 4` = **True** | `continue` — skip | 0 |
| 2 | 5002 | 5 | `5 < 4` = **False** | append, `1 >= 3` = False | 1 |
| 3 | 5003 | 4 | `4 < 4` = **False** | append, `2 >= 3` = False | 2 |
| 4 | 5004 | 2 | `2 < 4` = **True** | `continue` — skip | 2 |
| 5 | 5005 | 5 | `5 < 4` = **False** | append, `3 >= 3` = **True** → `break` | 3 |
| — | 5006 | 4 | **never reached** | loop already exited | — |

> `continue` skips the rest of the **current iteration** and moves to the next.  
> `break` exits the **entire loop** — items after the break point are never visited.


---


## 6 — List Comprehension

A list comprehension is the **accumulation pattern rewritten in one line**.

```python
# Standard loop (accumulation):
results = []
for item in collection:
    if condition:
        results.append(transform(item))

# List comprehension — same result:
results = [transform(item)  for item in collection  if condition]
#          ↑ WHAT            ↑ THE LOOP              ↑ FILTER (optional)
```

| Use | When |
|-----|------|
| List comprehension | Simple transform or filter on one collection |
| Regular for loop | Complex logic, multiple steps, side effects |


In [ ]:
# squares = [n*n for n in range(1, 6)]
# TRACE: n=1 → 1*1=1   include
#        n=2 → 2*2=4   include
#        n=3 → 3*3=9   include
#        n=4 → 4*4=16  include
#        n=5 → 5*5=25  include
#        result → [1, 4, 9, 16, 25]
squares = [n * n  for n in range(1, 6)]

# evens = [n for n in range(1, 11) if n%2==0]
# TRACE: n=1 → 1%2=1 ≠ 0 → skip  | n=2 → 2%2=0 → include
#        n=3 → 3%2=1 ≠ 0 → skip  | n=4 → 4%2=0 → include
#        n=5 → skip | n=6 → include | n=7 → skip | n=8 → include
#        n=9 → skip | n=10 → include
#        result → [2, 4, 6, 8, 10]
evens   = [n  for n in range(1, 11)  if n % 2 == 0]

# upper = [w.upper() for w in words]
# TRACE: w="hello"  → "HELLO"  | w="world"  → "WORLD"  | w="python" → "PYTHON"
#        result → ["HELLO", "WORLD", "PYTHON"]
upper   = [w.upper()  for w in ["hello", "world", "python"]]

print("Squares:", squares)
print("Evens:  ", evens)
print("Upper:  ", upper)


In [ ]:
products = [
    {"name": "Classic Monitor",   "price": 205.21, "stock": 238},
    {"name": "Ultimate Perfume",  "price": 568.17, "stock": 10},
    {"name": "Budget Headphones", "price": 29.99,  "stock": 0},
    {"name": "Yoga Mat Pro",      "price": 45.00,  "stock": 150},
    {"name": "Luxury Cream",      "price": 89.00,  "stock": 0},
]

# Filter — in_stock = [p for p in products if p["stock"] > 0]
# TRACE: Classic Monitor   stock=238 → 238>0 = True  → include
#        Ultimate Perfume   stock=10  → 10>0  = True  → include
#        Budget Headphones  stock=0   → 0>0   = False → skip
#        Yoga Mat Pro       stock=150 → 150>0 = True  → include
#        Luxury Cream       stock=0   → 0>0   = False → skip
#        result → [Classic Monitor, Ultimate Perfume, Yoga Mat Pro]
in_stock = [p for p in products  if p["stock"] > 0]
print("In stock:", [p["name"] for p in in_stock])

# Filter + transform — affordable in-stock → formatted prompt lines
# TRACE: Classic Monitor   stock=238>0 True, price=205.21<300 True  → include → "- Classic Monitor ($205.21)"
#        Ultimate Perfume   stock=10>0  True, price=568.17<300 False → skip
#        Budget Headphones  stock=0>0   False                        → skip
#        Yoga Mat Pro       stock=150>0 True, price=45.00<300 True   → include → "- Yoga Mat Pro ($45.00)"
#        Luxury Cream       stock=0>0   False                        → skip
#        result → ["- Classic Monitor ($205.21)", "- Yoga Mat Pro ($45.00)"]
prompt_lines = [
    f"- {p['name']} (${p['price']:.2f})"
    for p in products
    if p["stock"] > 0 and p["price"] < 300
]
print("Prompt lines:")
for line in prompt_lines:
    print(" ", line)


In [ ]:
# High-rated review titles — for few-shot examples
high_rated_titles = [
    r["title"]
    for r in all_reviews
    if r["rating"] >= 4
]
print("Few-shot candidates:", high_rated_titles)


---


## 7 — Dict Comprehension

A dict comprehension builds a `{key: value}` dict from a loop in one line.

```python
{key_expr: value_expr  for item in collection  if condition}
```

**Primary use:** turn a list into a fast O(1) lookup dict — instead of scanning the list every time (O(n)).


In [ ]:
# squares_dict = {n: n*n for n in range(1, 6)}
# TRACE: n=1 → key=1, value=1*1=1   → {1: 1}
#        n=2 → key=2, value=2*2=4   → {1:1, 2:4}
#        n=3 → key=3, value=3*3=9   → {1:1, 2:4, 3:9}
#        n=4 → key=4, value=4*4=16  → {1:1, 2:4, 3:9, 4:16}
#        n=5 → key=5, value=5*5=25  → {1:1, 2:4, 3:9, 4:16, 5:25}
squares_dict = {n: n*n        for n in range(1, 6)}

# word_lengths = {w: len(w) for w in words}
# TRACE: w="apple"  → key="apple",  value=5  → {"apple": 5}
#        w="banana" → key="banana", value=6  → {"apple":5, "banana":6}
#        w="cherry" → key="cherry", value=6  → {"apple":5, "banana":6, "cherry":6}
word_lengths  = {w: len(w)    for w in ["apple", "banana", "cherry"]}

print(squares_dict)            # {1:1, 2:4, 3:9, 4:16, 5:25}
print(squares_dict[3])         # 9 — instant O(1) lookup
print(word_lengths)            # {'apple': 5, 'banana': 6, 'cherry': 6}


In [ ]:
# product_index = {p["name"]: p for p in products}
# TRACE: p=Classic Monitor   → key="Classic Monitor",   value={...238 stock...}
#        p=Ultimate Perfume   → key="Ultimate Perfume",  value={...10 stock...}
#        p=Budget Headphones  → key="Budget Headphones", value={...0 stock...}
#        p=Yoga Mat Pro       → key="Yoga Mat Pro",      value={...150 stock...}
#        p=Luxury Cream       → key="Luxury Cream",      value={...0 stock...}
#        result → dict with 5 keys, each key points to the full product dict
product_index = {p["name"]: p  for p in products}

classic = product_index["Classic Monitor"]   # O(1) — no scanning
print(f"Price: {classic['price']}  Stock: {classic['stock']}")

# ratings_by_id = {r["review_id"]: r["rating"] for r in reviews}
# TRACE: r={5001,3,...} → key=5001, value=3
#        r={5002,5,...} → key=5002, value=5
#        r={5003,4,...} → key=5003, value=4
#        r={5004,2,...} → key=5004, value=2
#        result → {5001:3, 5002:5, 5003:4, 5004:2}
ratings_by_id = {r["review_id"]: r["rating"]  for r in reviews}
print(ratings_by_id)
print(f"Review 5002 rating: {ratings_by_id[5002]}")


In [ ]:
# Routing results → lookup dict
routing_index = {r["query"]: r["agent"] for r in results}

for query, agent in list(routing_index.items())[:3]:
    print(f"{query[:40]:40s} → {agent}")


---


## Summary — Day 02

| Concept | Key rule / syntax |
|---------|------------------|
| `if/elif/else` | Most urgent condition first. Always write `else`. Colon required. |
| Comparison ops | `== != > < >= <=` — produce `True` or `False` |
| `in` operator | `"cancel" in q` — substring or membership check |
| `and or not` | Combine conditions. `.lower()` before string matching. |
| `for` loop | `for item in collection:` — known collection |
| `enumerate()` | `for i, item in enumerate(col):` — index + value |
| Accumulation | `results = []` then `results.append(x)` inside loop |
| `range()` | `range(n)` · `range(start,stop)` · `range(start,stop,step)` |
| `while` loop | Unknown iterations. **Always update the condition variable.** |
| Exponential backoff | `wait = 0.1 * (2 ** attempt)` — standard LLM retry |
| `break` | Exit loop entirely. Used: enough collected / API succeeded. |
| `continue` | Skip this iteration. Used: filter out unwanted items. |
| List comprehension | `[expr for item in col if cond]` — filter or transform |
| Dict comprehension | `{key: val for item in col}` — build O(1) lookup index |

**Next:** Day 03 — Functions and Data Structures  
`def`, `*args`, `**kwargs`, all list/dict/set/tuple methods, `sorted()`, `map()`, `filter()`
